# All-Species CCF Validation — Retrieval 2148796  (De Regt+2024 §4.2)

**Goal**: Validate the detection of each trace species from retrieval
`2148796_N800_ev0.5_NormNone_PerChipScaleFalse` using the De Regt et al. (2024) §4.2 CCF methodology.

**Strategy** (De Regt+2024 §4.2):
1. Generate templates for the 4 free-chemistry trace species from the **MAP** (best-fit) parameters.
   - **Fiducial model** (`flux_all`): full-spectrum MAP model at barycentric rv.
   - **No-X model** (`flux_noX`): fiducial with species X mass fraction zeroed out.
   - **X template** (`flux_tmpl`): fiducial − noX  ← pure spectral signature of X.
2. Form **data residuals**: R = obs − noX  ← signal *containing* species X.
3. Apply per-chip Gaussian HP filter (σ = 300 px) to **both** R and M(rv) to remove broad continuum.
4. Cross-correlate template against data residuals (covariance-weighted):
   `CCF(rv) = Σ_{chips} Σᵢ  HP(M(rv))ᵢ × R_hp,i / σᵢ²`
5. Compute ACF: same sum with R_hp replaced by HP(M(0)):
   `ACF(rv) = Σ_{chips} Σᵢ  HP(M(rv))ᵢ × HP(M(0))ᵢ / σᵢ²`
6. SNR = max(CCF) / std(CCF − ACF outside ±200 km/s).
   - Signal: peak of CCF (always positive for a real detection).
   - Noise: std of CCF−ACF wings where ACF ≈ 0, representing the pure noise floor.
   - CCF ≈ ACF at the peak velocity indicates detected abundance ≈ model prediction.

## 1. Imports & Paths

In [1]:
import sys
import pickle
import importlib.util
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.ndimage import gaussian_filter1d

matplotlib.rcParams.update({'font.size': 12, 'figure.dpi': 120})

retrieval_series = '3766037'

RECIPE_DIR    = Path('/data2/peng/Recipe_DH_Tau_B')
WORKPATH      = Path('/data2/peng')
RETRIEVAL_ID  = '%s_N800_ev0.5_NormNone_PerChipScaleFalse' % retrieval_series
RETRIEVAL_DIR = WORKPATH / 'retrievals' / RETRIEVAL_ID
COMBINED_DIR  = WORKPATH / 'combined_two_nights'

sys.path.insert(0, str(RECIPE_DIR))

# Dynamic import of Guidebook v2.0
_guidebook_path = str(RECIPE_DIR / 'Guidebook_GAStronomy_Piette_v2.0.py')
_spec = importlib.util.spec_from_file_location('Guidebook_GAStronomy_Piette_v2_0', _guidebook_path)
_gb   = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_gb)

Target                        = _gb.Target
Parameters                    = _gb.Parameters
Retrieval                     = _gb.Retrieval
_load_night                   = _gb._load_night
make_free_params_free_chem    = _gb.make_free_params_free_chem
make_free_params_equilibrium  = _gb.make_free_params_equilibrium   # v628541: equil. chemistry
pRT_spectrum                  = _gb.pRT_spectrum

print('Retrieval dir:', RETRIEVAL_DIR)
print('Imports OK')


Input data path changed to '/net/lem/data2/pRT3_formatted/input_data'
Retrieval dir: /data2/peng/retrievals/3766037_N800_ev0.5_NormNone_PerChipScaleFalse
Imports OK


## 2. Load Best-Fit Parameters from Retrieval 2148796

In [2]:
with open(RETRIEVAL_DIR / 'final_params_dict.pickle', 'rb') as f:
    best_fit_params = pickle.load(f)

print('Best-fit parameters:')
for k, v in best_fit_params.items():
    if np.ndim(v) == 0:
        print(f'  {k:25s} = {float(v):.5g}')
VCORR=[-15.37101, -15.56411] #km/s
RV_SYS = [float(best_fit_params['rv_N1'])+VCORR[0], float(best_fit_params['rv_N2'])+VCORR[1]]  # ~ 31.6 km/s
print(f'\nSystem RV (rv_N1) = {RV_SYS[0]:.3f} km/s')
print(f'System RV (rv_N2) = {RV_SYS[1]:.3f} km/s')

#correct the best-fit RVs into barycentric frame using RV_SYS
best_fit_params['rv_N1'] = RV_SYS[0]
best_fit_params['rv_N2'] = RV_SYS[1]


Best-fit parameters:
  rv_N1                     = 31.612
  rv_N2                     = 31.63
  vsini                     = 7.4708
  epsilon                   = 0.86496
  log_M                     = 1.0613
  log_R                     = 0.47148
  T_anchor                  = 2175.6
  dT_1                      = 472.74
  dT_2                      = 436.03
  dT_3                      = 77.697
  dT_4                      = 178.6
  dT_5                      = 236.74
  dT_6                      = 242.84
  dT_7                      = 301.37
  C_H                       = -0.46304
  C/O                       = 0.60301
  log_12CO_13CO             = 1.3535
  log_X_MgSiO3              = -0.53327
  log_X_Fe                  = -1.2962
  fsed                      = 6.2934
  log_Kzz                   = 11.773
  sigma_lnorm               = 1.6167
  [C/H]                     = -0.44151
  [C/H]_xsolar              = 0.36181
  s2                        = 1
  chi2                      = 4.5947
  chi2_N2    

## 3. Spectrum 1 — Load Existing Retrieval Model

In [3]:
wave1_raw = np.load(RETRIEVAL_DIR / 'retrieval_model_wave.npy')
flux1_raw = np.load(RETRIEVAL_DIR / 'retrieval_model_flux.npy')   # unscaled absolute flux

m = np.isfinite(wave1_raw) & np.isfinite(flux1_raw)
wave1, flux1 = wave1_raw[m], flux1_raw[m]
idx = np.argsort(wave1)
wave1, flux1 = wave1[idx], flux1[idx]

print(f'Spectrum 1: {len(wave1)} pixels,  λ = {wave1.min():.1f}–{wave1.max():.1f} nm')

Spectrum 1: 24950 pixels,  λ = 2063.9–2470.7 nm


## 4. Forward Modeling Setup

Instantiate the `Retrieval` object (mirrors `tasting_retrieval_equa_chem_v5.0_piette_RM.py`).  
This initialises the pRT3 Radtrans opacity tables (≈ 2–5 min). **Do not call `retrieval.run_retrieval()`.**

In [4]:
Normalize_method  = None   # NormNone — absolute flux mode
scaling_parameter = False
use_absolute_flux = True

wave_N1, flux_N1, err_N1, R_N1 = _load_night(
    '2022-12-31',
    flux_file        = 'extracted_spectra_combined_flux_cal.npy',
    err_file         = 'extracted_spectra_combined_err_flux_cal.npy',
    normalize_method = Normalize_method,
)
wave_N2, flux_N2, err_N2, R_N2 = _load_night(
    '2023-01-01',
    flux_file        = 'extracted_spectra_combined_flux_cal.npy',
    err_file         = 'extracted_spectra_combined_err_flux_cal.npy',
    normalize_method = Normalize_method,
)

T1 = Target(wl=wave_N1, fl=flux_N1, err=err_N1, name='dh_tau_b_N1')
T2 = Target(wl=wave_N2, fl=flux_N2, err=err_N2, name='dh_tau_b_N2')

# Retrieval 628541 used equilibrium chemistry.
# Using equilibrium chemistry here ensures H2O (and all other species) are computed
# self-consistently from the best-fit C_H, C/O, log_12CO_13CO rather than from a
# random free-VMR cube initialization — which would give the wrong template strength.
constant_params = {'chemistry': 'equilibrium'}
free_params     = make_free_params_equilibrium()
parameters      = Parameters(free_params, constant_params)

# Initialise parameter dict with a dummy cube, then overwrite with best-fit values
parameters(np.random.rand(parameters.ndim))
parameters.params.update(best_fit_params)

retrieval = Retrieval(
    parameters         = parameters,
    N_live_points      = 800,
    evidence_tolerance = 0.5,
    targets            = [T1, T2],
    testing            = False,
    normalize_flux     = Normalize_method,
    per_chip_scaling   = scaling_parameter,
    instrument_res     = [R_N1, R_N2],
    use_absolute_flux  = use_absolute_flux,
)
# Make sure best-fit params are set (Retrieval.__init__ may call parameters())
retrieval.parameters.params.update(best_fit_params)

print('Retrieval atmosphere object ready.')


  [2022-12-31] No normalisation applied.
  [2022-12-31] Estimating resolving power...
  Estimated R = 314490 (median over 15 chips, range 297831–333720)
  [2022-12-31] Valid pixels: 24950 / 30720
  [2023-01-01] No normalisation applied.
  [2023-01-01] Estimating resolving power...
  Estimated R = 314480 (median over 15 chips, range 297860–333843)
  [2023-01-01] Valid pixels: 24964 / 30720
[Target dh_tau_b_N1] wl:(24950,)  fl:(24950,)  err:(24950,)
[Target dh_tau_b_N2] wl:(24964,)  fl:(24964,)  err:(24964,)
Instrument resolving power per night: [314489, 314480]
Loading equilibrium chemistry table (done once)...
Loading chemical equilibrium chemistry table from file '/net/lem/data2/pRT3_formatted/input_data/pre_calculated_chemistry/equilibrium_chemistry/equilibrium_chemistry.chemtable.petitRADTRANS.h5'... Done.
Equilibrium chemistry table loaded.
Creating new atmosphere object...
Loading Radtrans opacities...
 Loading line opacities of species '1H2-16O' from file '/net/lem/data2/pRT3_for

## 5. Generate Full-Model Spectrum (Spectrum 1, Barycentric RV)


In [5]:
# use_absolute_flux must be passed explicitly — the default on pRT_spectrum is False.
# Without it, make_spectrum() returns the raw pRT3 surface flux (~1e5 W/m²/μm)
# instead of the (R/d)²-scaled observed flux (~1e-15 W/m²/μm).
#UAF = retrieval.use_absolute_flux   # True for retrieval 2148796
UAF = True

# Re-generate Spectrum 1 with the BARYCENTRIC rv_N1.
# The file retrieval_model_flux.npy was saved using the TOPOCENTRIC rv_N1
# (~31.62 km/s), but best_fit_params['rv_N1'] was corrected to the barycentric
# frame in Cell 4 (~16.25 km/s). All per-species templates must share the same
# velocity reference.
prt_full = pRT_spectrum(retrieval, spectral_resolution=100_000,
                         use_absolute_flux=UAF)
print('mass_fractions keys:', list(prt_full.mass_fractions.keys()))

flux1_bary = prt_full.make_spectrum(data_wave=wave1, rv_key='rv_N1')
frac_diff  = np.nanmax(np.abs(flux1_bary - flux1) / (np.abs(flux1) + 1e-40))
print(f'Spectrum 1 bary vs file: max fractional diff = {frac_diff:.3e}'
      f'  (expected large — ~15 km/s RV shift)')
print(f'  flux1_bary range: [{np.nanmin(flux1_bary):.3e}, {np.nanmax(flux1_bary):.3e}]')


mass_fractions keys: ['1H2-16O', '12C-16O', '13C-16O', '12C-1H4__MM', '14N-1H3', '1H2-32S', '1H-12C-14N', '12C-16O2__HITEMP', '56Fe-1H', 'H2', 'He', 'MMW']
Spectrum 1 bary vs file: max fractional diff = 1.196e+00  (expected large — ~15 km/s RV shift)
  flux1_bary range: [2.843e-16, 1.405e-15]


## 6. Species Configuration & Template Generator


In [6]:
# pRT species name -> human-readable label for plots
# Only the 4 species retrieved by 2148796 (free chemistry).
# NH3, H2S, HCN, CO2, FeH are absent from the free-chem Radtrans atmosphere
# and would cause a KeyError / pRT error when zeroed in make_noX_template.
TRACE_SPECIES = {
    '1H2-16O':     'H2O',
    '12C-16O':     '12CO',
    '13C-16O':     '13CO',
    '12C-1H4__MM': 'CH4',
}


def make_noX_template(retrieval, flux_all, wave, rv_key, species_key, use_absolute_flux):
    """Return (flux_noX, flux_tmpl_X) for species `species_key`.

    flux_noX   : model spectrum with species_key mass fraction zeroed.
    flux_tmpl_X: flux_all - flux_noX  (<=0 at species X absorption lines).

    Using the spectral difference as the CCF template avoids the continuum-shape
    artefact that arises when running RT with only species X and H2/He background
    (other absorbers removed -> deeper, hotter photosphere exposed -> spurious
    correlations at the wrong velocity).
    """
    prt_noX = pRT_spectrum(retrieval, spectral_resolution=100_000,
                           use_absolute_flux=use_absolute_flux)
    prt_noX.mass_fractions[species_key] = np.zeros(retrieval.n_atm_layers)
    prt_noX._prt_wl   = None   # clear pRT flux cache after modifying mass_fractions
    prt_noX._prt_flux = None
    flux_noX  = prt_noX.make_spectrum(data_wave=wave, rv_key=rv_key)
    flux_tmpl = flux_all - flux_noX
    return flux_noX, flux_tmpl


print(f'TRACE_SPECIES: {list(TRACE_SPECIES.values())}')
print('make_noX_template() defined.')


TRACE_SPECIES: ['H2O', '12CO', '13CO', 'CH4']
make_noX_template() defined.


In [7]:
# Template generation loop — one pRT radiative-transfer call per species.
# Run once; results stored in `templates` dict keyed by pRT species name.
templates = {}
for prt_name, label in TRACE_SPECIES.items():
    print(f'  [{label}] generating no-X and X-only templates...')
    flux_noX, flux_tmpl = make_noX_template(
        retrieval, flux1_bary, wave1, 'rv_N1', prt_name, UAF)
    templates[prt_name] = dict(label=label, flux_noX=flux_noX, flux_tmpl=flux_tmpl)
    n_nonzero = np.sum(np.abs(flux_tmpl) > 1e-20)
    print(f'    flux_tmpl range [{np.nanmin(flux_tmpl):.2e}, {np.nanmax(flux_tmpl):.2e}]'
          f'  non-zero px (>1e-20): {n_nonzero}')
print('\nAll templates ready.')


  [H2O] generating no-X and X-only templates...
    flux_tmpl range [-1.11e-15, -3.17e-18]  non-zero px (>1e-20): 24950
  [12CO] generating no-X and X-only templates...
    flux_tmpl range [-5.17e-16, -3.28e-27]  non-zero px (>1e-20): 8401
  [13CO] generating no-X and X-only templates...
    flux_tmpl range [-1.77e-16, -7.89e-31]  non-zero px (>1e-20): 6008
  [CH4] generating no-X and X-only templates...
    flux_tmpl range [-5.96e-20, -2.06e-23]  non-zero px (>1e-20): 9811

All templates ready.


## 7. Plot — Spectra Comparison

In [8]:
# Show full range and a 13CO-rich zoom region (~2330-2350 nm).
# Use templates dict for variables so no separate pRT call is needed.
co13_tmpl    = templates['13C-16O']
flux2        = co13_tmpl['flux_noX']
flux3_template = co13_tmpl['flux_tmpl']

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

ax = axes[0]
ax.plot(wave1, flux1_bary, lw=0.6, alpha=0.8, label='Spectrum 1 (all species)')
ax.plot(wave1, flux2,      lw=0.6, alpha=0.8, label='Spectrum 2 (no 13CO)')
ax.set_xlabel('Wavelength (nm)')
ax.set_ylabel('Flux')
ax.set_title('Model Spectra — Full K-band')
ax.legend(fontsize=10)

ax = axes[1]
zoom_lo, zoom_hi = 2330, 2355
for w, f, lbl in [(wave1, flux1_bary, 'Sp.1 all'), (wave1, flux2, 'Sp.2 no 13CO')]:
    mask = (w >= zoom_lo) & (w <= zoom_hi)
    ax.plot(w[mask], f[mask], lw=1.0, label=lbl)
diff_mask = (wave1 >= zoom_lo) & (wave1 <= zoom_hi)
ax.plot(wave1[diff_mask], flux3_template[diff_mask] * 10,
        lw=1.0, ls='--', color='red',
        label='(Sp.1 - Sp.2) x10  [13CO CCF template]')
ax.set_xlabel('Wavelength (nm)')
ax.set_ylabel('Flux')
ax.set_title(f'Zoom: {zoom_lo}-{zoom_hi} nm  (13CO R-branch)')
ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig(RETRIEVAL_DIR / 'validation_spectra_comparison_CO.png', dpi=150)
plt.show()

#Same codes for CH4
# Show full range and a 1CH4-rich zoom region (~2330-2350 nm).
# Use templates dict for variables so no separate pRT call is needed.
ch4_tmpl    = templates['12C-1H4__MM']
flux2        = ch4_tmpl['flux_noX']
flux3_template = ch4_tmpl['flux_tmpl']

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

ax = axes[0]
ax.plot(wave1, flux1_bary, lw=0.6, alpha=0.8, label='Spectrum 1 (all species)')
ax.plot(wave1, flux2,      lw=0.6, alpha=0.8, label='Spectrum 2 (no 13CO)')
ax.set_xlabel('Wavelength (nm)')
ax.set_ylabel('Flux')
ax.set_title('Model Spectra — Full K-band')
ax.legend(fontsize=10)

ax = axes[1]
zoom_lo, zoom_hi = 2330, 2355
for w, f, lbl in [(wave1, flux1_bary, 'Sp.1 all'), (wave1, flux2, 'Sp.2 no 13CO')]:
    mask = (w >= zoom_lo) & (w <= zoom_hi)
    ax.plot(w[mask], f[mask], lw=1.0, label=lbl)
diff_mask = (wave1 >= zoom_lo) & (wave1 <= zoom_hi)
ax.plot(wave1[diff_mask], flux3_template[diff_mask] * 10,
        lw=1.0, ls='--', color='red',
        label='(Sp.1 - Sp.2) x10  [13CO CCF template]')
ax.set_xlabel('Wavelength (nm)')
ax.set_ylabel('Flux')
ax.set_title(f'Zoom: {zoom_lo}-{zoom_hi} nm  (13CO R-branch)')
ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig(RETRIEVAL_DIR / 'validation_spectra_comparison_CH4.png', dpi=150)
plt.show()



/var/tmp/peng/ipykernel_32999/1836448557.py:33: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/tmp/peng/ipykernel_32999/1836448557.py:68: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 8. Load Absolute-Calibrated Two-Night Combined Spectrum

In [9]:
# Per-night flux-calibrated data — in absolute flux units (W/m²/nm), matching the model templates.
# The combined two-night file (extracted_spectra_combined_two_nights.npy) is in normalised units
# (~1e-3 vs model ~1e-15).  Using it for R = obs − noX makes noX negligible, so R ≈ obs_raw,
# ACF << CCF by ~1e12, and the CCF peak drifts to a random velocity.
# Per-night flux_cal files share the same absolute unit as pRT, so CCF ≈ ACF at the detection peak.

NIGHT1_DIR = Path('/data2/peng/2022-12-31')
NIGHT2_DIR = Path('/data2/peng/2023-01-01')

obs_flux_N1 = np.load(NIGHT1_DIR / 'extracted_spectra_combined_flux_cal.npy').astype(float)
obs_flux_N2 = np.load(NIGHT2_DIR / 'extracted_spectra_combined_flux_cal.npy').astype(float)
err_N1      = np.load(NIGHT1_DIR / 'extracted_spectra_combined_err_flux_cal.npy').astype(float)
err_N2      = np.load(NIGHT2_DIR / 'extracted_spectra_combined_err_flux_cal.npy').astype(float)
obs_wave_N1 = np.load(NIGHT1_DIR / 'barycentric_wavelengths_night1.npy').astype(float)
obs_wave_N2 = np.load(NIGHT2_DIR / 'barycentric_wavelengths_night2.npy').astype(float)

nDet, nOrder, nPix = obs_wave_N1.shape
print(f'Per-night flux_cal: ({nDet} det, {nOrder} orders, {nPix} px)')
print(f'  Night 1 flux range: [{np.nanmin(obs_flux_N1):.2e}, {np.nanmax(obs_flux_N1):.2e}]')
print(f'  Night 2 flux range: [{np.nanmin(obs_flux_N2):.2e}, {np.nanmax(obs_flux_N2):.2e}]')
print(f'  Night 1 err  range: [{np.nanmin(err_N1[err_N1>0]):.2e}, {np.nanmax(err_N1):.2e}]')
print(f'  Night 2 err  range: [{np.nanmin(err_N2[err_N2>0]):.2e}, {np.nanmax(err_N2):.2e}]')

# Inverse variance per night: non-positive / non-finite err → 0 (no contribution to CCF sum)
ivar_N1 = np.where(np.isfinite(err_N1) & (err_N1 > 0), 1.0 / err_N1**2, 0.0)
ivar_N1[~np.isfinite(obs_flux_N1)] = 0.0
ivar_N2 = np.where(np.isfinite(err_N2) & (err_N2 > 0), 1.0 / err_N2**2, 0.0)
ivar_N2[~np.isfinite(obs_flux_N2)] = 0.0
print(f'  ivar_N1 non-zero px: {(ivar_N1 > 0).sum()}  |  ivar_N2 non-zero px: {(ivar_N2 > 0).sum()}')

valid_mask_N1 = np.isfinite(obs_wave_N1)
valid_mask_N2 = np.isfinite(obs_wave_N2)

# obs_wave_3d = night 1 wavelength grid (identical to WLEN_combined_two_nights_bary.npy)
obs_wave_3d = obs_wave_N1
valid_mask  = valid_mask_N1

# Flat 1-D arrays for plotting only (use night 1 flux_cal)
obs_wave_flat = obs_wave_N1.reshape(-1)
obs_flux_flat = obs_flux_N1.reshape(-1)
good = np.isfinite(obs_wave_flat) & np.isfinite(obs_flux_flat)
obs_wave_flat, obs_flux_flat = obs_wave_flat[good], obs_flux_flat[good]
idx = np.argsort(obs_wave_flat)
obs_wave_flat, obs_flux_flat = obs_wave_flat[idx], obs_flux_flat[idx]
print(f'Flat (for plotting, night 1): {len(obs_wave_flat)} valid pixels')

Per-night flux_cal: (3 det, 5 orders, 2048 px)
  Night 1 flux range: [-8.82e-16, 2.72e-15]
  Night 2 flux range: [-9.88e-16, 2.21e-15]
  Night 1 err  range: [8.86e-17, 2.89e-15]
  Night 2 err  range: [6.28e-17, 2.32e-15]
  ivar_N1 non-zero px: 24950  |  ivar_N2 non-zero px: 24964
Flat (for plotting, night 1): 24950 valid pixels


## 9. De Regt §4.2 CCF — Helpers

In [10]:
C_KMS = 2.99792458e5   # speed of light in km/s


def highpass_filter_chips(flux_3d, sigma_px=300):
    """Per-chip Gaussian HP filter (De Regt+2024 §4.2, σ=300 px).
    Subtracts Gaussian-smoothed continuum from each chip row independently.
    NaN pixels are zeroed before smoothing and zeroed in the output.
    """
    out = np.zeros_like(flux_3d)
    for d in range(flux_3d.shape[0]):
        for o in range(flux_3d.shape[1]):
            row    = flux_3d[d, o].copy()
            finite = np.isfinite(row)
            if finite.sum() < 10:
                continue
            tmp       = np.where(finite, row, 0.0)
            out[d, o] = np.where(finite, tmp - gaussian_filter1d(tmp, sigma=sigma_px), 0.0)
    return out


def _hp_chip(arr, fm, sigma_px):
    """HP-filter a single chip array in-place-safe. Invalid pixels → 0."""
    tmp = np.where(fm, arr, 0.0)
    return np.where(fm, tmp - gaussian_filter1d(tmp, sigma=sigma_px), 0.0)


def run_ccf_acf_deregt(wave_1d, flux_tmpl, obs_wave_3d, R_hp_3d, ivar_3d, rvlag, sigma_px=300):
    """CCF and ACF with HP filter applied to BOTH M and R (De Regt+2024 §4.2).

    CCF(rv) = Σ_chips  HP(M(rv)) × R_hp / σ²
    ACF(rv) = Σ_chips  HP(M(rv)) × HP(M(0)) / σ²

    HP filter on M ensures the broad continuum component of the spectral difference
    (fiducial − noX) is removed before correlation, so ACF and CCF share the same
    frequency content as R_hp. Without HP on M the ACF has a much larger amplitude
    than the CCF (continuum correlates with itself but not with HP-filtered R),
    making CCF − ACF wrongly negative everywhere.
    """
    nDet, nOrder, _ = obs_wave_3d.shape

    # Precompute per-chip quantities that are fixed across rv steps

    chip_wave = [[obs_wave_3d[d, o]                               for o in range(nOrder)] for d in range(nDet)]
    chip_fm   = [[np.isfinite(obs_wave_3d[d, o])                  for o in range(nOrder)] for d in range(nDet)]
    chip_R    = [[np.where(np.isfinite(R_hp_3d[d, o]), R_hp_3d[d, o], 0.0)
                  for o in range(nOrder)] for d in range(nDet)]
    chip_iv   = [[ivar_3d[d, o]                                   for o in range(nOrder)] for d in range(nDet)]

    # HP-filtered template at rv=0 (ACF partner)
    chip_M0hp = [[_hp_chip(np.interp(chip_wave[d][o], wave_1d, flux_tmpl, left=0., right=0.),
                            chip_fm[d][o], sigma_px)
                  for o in range(nOrder)] for d in range(nDet)]

    nrv = len(rvlag)
    ccf = np.zeros(nrv)
    acf = np.zeros(nrv)

    for i, rv in enumerate(rvlag):
        factor  = 1.0 + rv / C_KMS
        ccf_sum = 0.0
        acf_sum = 0.0
        for d in range(nDet):
            for o in range(nOrder):
                m    = np.interp(chip_wave[d][o] / factor, wave_1d, flux_tmpl, left=0., right=0.)
                mhp  = _hp_chip(m, chip_fm[d][o], sigma_px)
                wiv  = mhp * chip_iv[d][o]
                ccf_sum += float(np.dot(wiv, chip_R[d][o]))
                acf_sum += float(np.dot(wiv, chip_M0hp[d][o]))
        ccf[i] = ccf_sum
        acf[i] = acf_sum
    return ccf, acf


def snr_deregt(rvlag, ccf, acf, exclude_kms=200.0):
    """SNR = max(CCF) / std(CCF − ACF outside ±exclude_kms)  (De Regt+2024 §4.2).

    Signal: peak of the CCF itself (positive for a real detection).
    Noise : std of the CCF − ACF residuals far from the peak, where the ACF has
            decayed to ~0 and the residual represents the pure noise floor.
    """
    residual   = ccf - acf
    noise_mask = np.abs(rvlag) > exclude_kms
    if noise_mask.sum() == 0:
        return np.nan, rvlag[np.nanargmax(ccf)], ccf[np.nanargmax(ccf)], np.nan, residual
    std_noise  = np.nanstd(residual[noise_mask])
    peak_idx   = np.nanargmax(ccf)          # peak of CCF, not CCF-ACF
    peak_rv    = rvlag[peak_idx]
    peak_val   = ccf[peak_idx]
    snr        = peak_val / std_noise if std_noise > 0 else np.nan
    return snr, peak_rv, peak_val, std_noise, residual


print('De Regt §4.2 CCF helpers defined (v2).')
print('  highpass_filter_chips  — per-chip Gaussian HP, σ=300 px')
print('  run_ccf_acf_deregt     — HP applied to BOTH M(rv) and R; correct ACF shape')
print('  snr_deregt             — signal=max(CCF), noise=std(CCF−ACF, |rv|>200 km/s)')


De Regt §4.2 CCF helpers defined (v2).
  highpass_filter_chips  — per-chip Gaussian HP, σ=300 px
  run_ccf_acf_deregt     — HP applied to BOTH M(rv) and R; correct ACF shape
  snr_deregt             — signal=max(CCF), noise=std(CCF−ACF, |rv|>200 km/s)


## 10. H₂O Sanity Check — Single Chip

H₂O is the strongest expected detection (log VMR = −4.03, dense K-band lines).
Run a quick ±200 km/s CCF on the single chip with the largest H₂O template RMS.
A clean peak near 0 km/s confirms the De Regt pipeline is correctly wired before
the full ±1000 km/s loop over all species.


In [11]:
# H2O template
h2o_key    = '1H2-16O'
flux_noH2O = templates[h2o_key]['flux_noX']
flux_tmH2O = templates[h2o_key]['flux_tmpl']

# Data residual for night 1: R = obs_flux_N1 − noH2O  (both in absolute flux units)
noH2O_3d_N1 = np.zeros_like(obs_wave_N1)
noH2O_3d_N1[valid_mask_N1] = np.interp(obs_wave_N1[valid_mask_N1], wave1, flux_noH2O,
                                        left=0.0, right=0.0)
R_hp_H2O = highpass_filter_chips(obs_flux_N1 - noH2O_3d_N1)

# Template at rv=0 on night-1 obs grid (used to identify the best chip)
M0_3d_raw = np.zeros_like(obs_wave_N1)
M0_3d_raw[valid_mask_N1] = np.interp(obs_wave_N1[valid_mask_N1], wave1, flux_tmH2O,
                                      left=0.0, right=0.0)

# Find the chip with the largest H2O template RMS (night 1 wavelength grid)
best_d, best_o, best_rms = 0, 0, 0.0
for d in range(nDet):
    for o in range(nOrder):
        rms = np.nanstd(M0_3d_raw[d, o])
        if rms > best_rms:
            best_rms, best_d, best_o = rms, d, o
print(f'Best chip for H2O: det={best_d}, order={best_o}  (template RMS = {best_rms:.2e})')

# Single-chip ±1000 km/s CCF using night 1 flux_cal
rvlag_test = np.arange(-1000.0, 1001.0, 1.0)
nrv_t      = len(rvlag_test)
ccf_t      = np.zeros(nrv_t)
acf_t      = np.zeros(nrv_t)

chip_w  = obs_wave_N1[best_d, best_o]
chip_fm = np.isfinite(chip_w)
chip_R  = np.where(np.isfinite(R_hp_H2O[best_d, best_o]), R_hp_H2O[best_d, best_o], 0.0)
chip_iv = ivar_N1[best_d, best_o]   # absolute-flux ivar for night 1

# HP-filtered M at rv=0 for ACF
chip_M0hp = _hp_chip(np.interp(chip_w, wave1, flux_tmH2O, left=0., right=0.), chip_fm, 300)

for i, rv in enumerate(rvlag_test):
    factor   = 1.0 + rv / C_KMS
    m        = np.interp(chip_w / factor, wave1, flux_tmH2O, left=0.0, right=0.0)
    mhp      = _hp_chip(m, chip_fm, 300)
    wiv      = mhp * chip_iv
    ccf_t[i] = float(np.dot(wiv, chip_R))
    acf_t[i] = float(np.dot(wiv, chip_M0hp))

snr_t, rv_peak_t, peak_val_t, std_n_t, resid_t = snr_deregt(rvlag_test, ccf_t, acf_t)
print(f'H2O single-chip: peak @ {rv_peak_t:+.1f} km/s   SNR = {snr_t:.2f}')
print(f'  σ_noise = {std_n_t:.3e}   CCF peak = {peak_val_t:.3e}')
print(f'  CCF/ACF at rv=0: {ccf_t[1000]:.3e} / {acf_t[1000]:.3e}  (should be ~1 if model abundance matches data)')
print(f'  (expected peak near 0 km/s if bary alignment is correct)')

fig = plt.figure(figsize=(16, 6))
gs  = fig.add_gridspec(2, 2, hspace=0.35, wspace=0.3)
ax1 = fig.add_subplot(gs[0, 0])
ax2 = fig.add_subplot(gs[1, 0], sharex=ax1)
ax3 = fig.add_subplot(gs[:, 1])

# Left col: De Regt Fig. 6 style (no scaling between CCF and ACF)
ax1.plot(rvlag_test, ccf_t, color='steelblue', lw=0.9, label='CCF')
ax1.plot(rvlag_test, acf_t, color='orange',    lw=0.9, ls='--', alpha=0.8, label='ACF')
ax1.axvline(rv_peak_t, color='k', ls=':', lw=0.9)
ax1.axhline(0, color='grey', lw=0.3)
ax1.set_ylabel('CC (absolute flux units)')
ax1.set_title(f'H₂O — CCF + ACF  (det={best_d}, order={best_o}, night 1)', fontsize=10)
ax1.legend(fontsize=8)

ax2.plot(rvlag_test, resid_t, color='crimson', lw=0.9)
ax2.axhline(0, color='grey', lw=0.3)
ax2.axvline(rv_peak_t, color='k', ls=':', lw=0.9)
ax2.set_ylabel('CCF − ACF')
ax2.set_xlabel('RV lag (km/s)')
ax2.set_title('CCF − ACF residuals', fontsize=10)

# Right col: CCF / σ
if std_n_t and std_n_t > 0:
    ax3.plot(rvlag_test, ccf_t / std_n_t, color='steelblue', lw=0.9)
    ax3.axhline(snr_t, color='tomato', lw=0.7, ls='--', label=f'SNR = {snr_t:.2f}')
ax3.axvline(rv_peak_t, color='k', ls=':', lw=0.9, label=f'peak @ {rv_peak_t:+.1f} km/s')
ax3.axhline(0, color='grey', lw=0.3)
ax3.axhline( 3, color='grey', lw=0.6, ls='--', alpha=0.4)
ax3.axhline(-3, color='grey', lw=0.6, ls='--', alpha=0.4)
ax3.set_ylabel('CCF / σ_noise')
ax3.set_xlabel('RV lag (km/s)')
ax3.set_title('CCF / σ   (σ from CCF−ACF, |rv|>200 km/s)', fontsize=10)
ax3.legend(fontsize=8)

fig.suptitle('H₂O sanity check (single chip, night 1, ±1000 km/s)', fontsize=12)
plt.savefig(RETRIEVAL_DIR / 'validation_deregt_H2O_sanity.png', dpi=150)
plt.show()

Best chip for H2O: det=0, order=4  (template RMS = 1.50e-16)
H2O single-chip: peak @ +1.0 km/s   SNR = 5.72
  σ_noise = 8.856e+01   CCF peak = 5.068e+02
  CCF/ACF at rv=0: 5.044e+02 / 3.999e+02  (should be ~1 if model abundance matches data)
  (expected peak near 0 km/s if bary alignment is correct)


/var/tmp/peng/ipykernel_32999/1580990345.py:91: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 11. Full De Regt CCF Loop — All Species (±1000 km/s)

Runs CCF and ACF for each retrieved species over ±1000 km/s at 1 km/s steps.
The wide range ensures the far wings (|rv| > 200 km/s) give a clean noise estimate.
Species with negligible template RMS (e.g. CH4 at these T/P) are skipped automatically.


In [12]:
rvlag_full = np.arange(-1000.0, 1001.0, 1.0)
print(f'RV grid: {rvlag_full[0]:.0f}–{rvlag_full[-1]:.0f} km/s  ({len(rvlag_full)} steps)')
print('Running CCF per night (both in absolute flux units) and summing.')

deregt_results = {}
for prt_name, tmpl in templates.items():
    label     = tmpl['label']
    flux_noX  = tmpl['flux_noX']
    flux_tmpl = tmpl['flux_tmpl']

    tmpl_rms = np.nanstd(flux_tmpl)
    if tmpl_rms < 1e-20:
        print(f'  [{label}] template RMS = {tmpl_rms:.1e} — skipping (no signal in K band)')
        continue

    print(f'  [{label}] CCF ±1000 km/s (night 1 + night 2)...', flush=True)

    # Night 1: R_N1 = obs_flux_N1 − noX  (both absolute units)
    noX_3d_N1 = np.zeros_like(obs_wave_N1)
    noX_3d_N1[valid_mask_N1] = np.interp(obs_wave_N1[valid_mask_N1], wave1, flux_noX,
                                          left=0.0, right=0.0)
    R_hp_N1 = highpass_filter_chips(obs_flux_N1 - noX_3d_N1)
    ccf_N1, acf_N1 = run_ccf_acf_deregt(wave1, flux_tmpl, obs_wave_N1, R_hp_N1, ivar_N1, rvlag_full)

    # Night 2: R_N2 = obs_flux_N2 − noX  (night 2 has a slightly different bary wavelength grid)
    noX_3d_N2 = np.zeros_like(obs_wave_N2)
    noX_3d_N2[valid_mask_N2] = np.interp(obs_wave_N2[valid_mask_N2], wave1, flux_noX,
                                          left=0.0, right=0.0)
    R_hp_N2 = highpass_filter_chips(obs_flux_N2 - noX_3d_N2)
    ccf_N2, acf_N2 = run_ccf_acf_deregt(wave1, flux_tmpl, obs_wave_N2, R_hp_N2, ivar_N2, rvlag_full)

    # Sum over nights (CCF is additive because it is a sum over pixels)
    ccf = ccf_N1 + ccf_N2
    acf = acf_N1 + acf_N2

    snr, peak_rv, peak_val, std_noise, residual = snr_deregt(rvlag_full, ccf, acf)
    deregt_results[prt_name] = dict(
        label=label, ccf=ccf, acf=acf, residual=residual,
        snr=snr, peak_rv=peak_rv, peak_val=peak_val, std_noise=std_noise,
    )
    zero_idx = np.argmin(np.abs(rvlag_full))
    print(f'    peak @ {peak_rv:+.1f} km/s   SNR = {snr:.2f}'
          f'   CCF/ACF @ v=0: {ccf[zero_idx]:.3e} / {acf[zero_idx]:.3e}')

print('\nDe Regt CCF loop complete.')
print(f'\n{"Species":<8}  {"SNR":>7}  {"Peak RV":>10}')
print('-' * 32)
for res in deregt_results.values():
    print(f'{res["label"]:<8}  {res["snr"]:>7.2f}  {res["peak_rv"]:>+10.1f} km/s')


RV grid: -1000–1000 km/s  (2001 steps)
Running CCF per night (both in absolute flux units) and summing.
  [H2O] CCF ±1000 km/s (night 1 + night 2)...
    peak @ +0.0 km/s   SNR = 35.75   CCF/ACF @ v=0: 1.699e+04 / 1.642e+04
  [12CO] CCF ±1000 km/s (night 1 + night 2)...
    peak @ +0.0 km/s   SNR = 22.51   CCF/ACF @ v=0: 4.177e+03 / 4.242e+03
  [13CO] CCF ±1000 km/s (night 1 + night 2)...
    peak @ -703.0 km/s   SNR = 3.45   CCF/ACF @ v=0: 1.272e+02 / 1.963e+02
  [CH4] CCF ±1000 km/s (night 1 + night 2)...
    peak @ +19.0 km/s   SNR = 2.94   CCF/ACF @ v=0: -1.776e-02 / 7.286e-05

De Regt CCF loop complete.

Species       SNR     Peak RV
--------------------------------
H2O         35.75        +0.0 km/s
12CO        22.51        +0.0 km/s
13CO         3.45      -703.0 km/s
CH4          2.94       +19.0 km/s


## 12. Plots — Per-Species CCF + SNR Summary

In [13]:
# Per-species two-column figure.
# Left column  (2 rows): De Regt Fig. 6 — CCF+ACF (top), CCF−ACF (bottom)
# Right column (1 row) : CCF / σ_noise   where σ = std(CCF−ACF, |rv|>200 km/s)
# CCF and ACF are plotted in raw units with no scaling between them, matching
# De Regt+2024 Fig. 6.  CCF ≈ ACF at v=0 means detected abundance ≈ model.

for prt_name, res in deregt_results.items():
    label    = res['label']
    ccf      = res['ccf']
    acf      = res['acf']
    residual = res['residual']   # CCF - ACF
    snr      = res['snr']
    peak_rv  = res['peak_rv']   # peak of CCF
    std_n    = res['std_noise']  # std(CCF-ACF outside ±200 km/s)

    fig = plt.figure(figsize=(16, 6))
    gs  = fig.add_gridspec(2, 2, hspace=0.35, wspace=0.3)
    ax1 = fig.add_subplot(gs[0, 0])
    ax2 = fig.add_subplot(gs[1, 0], sharex=ax1)
    ax3 = fig.add_subplot(gs[:, 1])

    # ── Left col, top: CCF + ACF (De Regt Fig. 6 upper panel) ──────────────
    ax1.plot(rvlag_full, ccf, color='steelblue', lw=0.8, label='CCF')
    ax1.plot(rvlag_full, acf, color='orange',    lw=0.8, ls='--', alpha=0.8, label='ACF')
    ax1.axvline(peak_rv, color='k', ls=':', lw=0.9, alpha=0.7)
    ax1.axhline(0, color='grey', lw=0.3)
    ax1.set_ylabel('CC')
    ax1.set_title(f'{label} — CCF + ACF', fontsize=10)
    ax1.legend(fontsize=8)
    plt.setp(ax1.get_xticklabels(), visible=False)

    # ── Left col, bottom: CCF − ACF (De Regt Fig. 6 lower panel) ───────────
    ax2.plot(rvlag_full, residual, color='crimson', lw=0.8)
    ax2.axhline(0, color='grey', lw=0.3)
    ax2.axvline(peak_rv, color='k', ls=':', lw=0.9, alpha=0.7)
    ax2.set_ylabel('CCF − ACF')
    ax2.set_xlabel('RV lag (km/s)')
    ax2.set_title('CCF − ACF residuals', fontsize=10)

    # ── Right col: CCF / σ_noise ─────────────────────────────────────────────
    ccf_norm = ccf / std_n if (std_n is not None and std_n > 0) else ccf
    ax3.plot(rvlag_full, ccf_norm, color='steelblue', lw=0.8)
    if snr is not None and not np.isnan(snr):
        ax3.axhline(snr, color='tomato', lw=0.7, ls='--', alpha=0.7,
                    label=f'SNR = {snr:.2f}')
    ax3.axvline(peak_rv, color='k', ls=':', lw=0.9, alpha=0.7,
                label=f'peak @ {peak_rv:+.1f} km/s')
    ax3.axhline(0, color='grey', lw=0.3)
    ax3.axhline( 3, color='grey', lw=0.6, ls='--', alpha=0.4)
    ax3.axhline(-3, color='grey', lw=0.6, ls='--', alpha=0.4)
    ax3.set_ylabel('CCF / σ_noise')
    ax3.set_xlabel('RV lag (km/s)')
    ax3.set_title('CCF / σ   (σ = std of CCF−ACF, |rv| > 200 km/s)', fontsize=10)
    ax3.legend(fontsize=8)

    fig.suptitle(f'Retrieval 2148796 — {label}  (De Regt+2024 §4.2)', fontsize=12)
    fname = RETRIEVAL_DIR / f'validation_deregt_{label}.png'
    plt.savefig(fname, dpi=150)
    plt.show()
    print(f'Saved: {fname.name}')


/var/tmp/peng/ipykernel_32999/1226475179.py:59: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Saved: validation_deregt_H2O.png
Saved: validation_deregt_12CO.png
Saved: validation_deregt_13CO.png
Saved: validation_deregt_CH4.png


In [14]:
# SNR summary bar chart
labels_plot = [res['label']   for res in deregt_results.values()]
snr_vals    = [res['snr']     for res in deregt_results.values()]
peak_rvs    = [res['peak_rv'] for res in deregt_results.values()]

palette    = ['steelblue', 'tomato', 'darkorange', 'mediumseagreen',
              'slategray', 'mediumpurple', 'peru', 'teal']
colors_bar = palette[:len(labels_plot)]

x = np.arange(len(labels_plot))
fig, ax = plt.subplots(figsize=(max(8, 2 * len(labels_plot)), 5))
bars = ax.bar(x, snr_vals, color=colors_bar, alpha=0.85, edgecolor='white', linewidth=0.5)
ax.bar_label(bars, labels=[f'{s:.1f}' for s in snr_vals], padding=3, fontsize=9)
ax.set_xticks(x)
ax.set_xticklabels(labels_plot, fontsize=11)
ax.set_ylabel('Detection SNR (De Regt §4.2)')
ax.set_title('All-Species CCF SNR — Retrieval 2148796\n'
             '(CCF−ACF peak / std outside ±200 km/s)')
ax.axhline(3, ls='--', color='grey', lw=0.9, label='SNR = 3')
ax.axhline(5, ls=':',  color='grey', lw=0.9, label='SNR = 5')

# Annotate peak RV below each bar
for xi, (snr, prv) in enumerate(zip(snr_vals, peak_rvs)):
    ax.text(xi, -0.3, f'{prv:+.1f} km/s', ha='center', va='top', fontsize=7, color='navy')

ax.set_ylim(bottom=min(0, min(snr_vals) - 1))
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig(RETRIEVAL_DIR / 'validation_deregt_snr_summary.png', dpi=150)
plt.show()
print('Saved: validation_deregt_snr_summary.png')


Saved: validation_deregt_snr_summary.png


/var/tmp/peng/ipykernel_32999/1351621520.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
